In [9]:
import duckdb

con = duckdb.connect("trips.duckdb")

print("Tables:")
print(con.execute("SHOW TABLES").df())

print("\nraw_trips schema:")
print(con.execute("DESCRIBE raw_trips").df())



Tables:
        name
0  raw_trips
1  raw_zones

raw_trips schema:
              column_name column_type null   key default extra
0                VendorID     INTEGER  YES  None    None  None
1    tpep_pickup_datetime   TIMESTAMP  YES  None    None  None
2   tpep_dropoff_datetime   TIMESTAMP  YES  None    None  None
3         passenger_count      BIGINT  YES  None    None  None
4           trip_distance      DOUBLE  YES  None    None  None
5              RatecodeID      BIGINT  YES  None    None  None
6      store_and_fwd_flag     VARCHAR  YES  None    None  None
7            PULocationID     INTEGER  YES  None    None  None
8            DOLocationID     INTEGER  YES  None    None  None
9            payment_type      BIGINT  YES  None    None  None
10            fare_amount      DOUBLE  YES  None    None  None
11                  extra      DOUBLE  YES  None    None  None
12                mta_tax      DOUBLE  YES  None    None  None
13             tip_amount      DOUBLE  YES  None    

In [2]:
print("\nraw_zones schema:")
print(con.execute("DESCRIBE raw_zones").df())


raw_zones schema:
    column_name column_type null   key default extra
0    LocationID      BIGINT  YES  None    None  None
1       Borough     VARCHAR  YES  None    None  None
2          Zone     VARCHAR  YES  None    None  None
3  service_zone     VARCHAR  YES  None    None  None


In [13]:
print(con.execute("""
    SELECT
        service_type,
        COUNT(*) AS total_rows,
        SUM(
            CASE
                WHEN service_type = 'yellow' AND tpep_pickup_datetime IS NULL THEN 1
                WHEN service_type = 'green'  AND lpep_pickup_datetime IS NULL THEN 1
                ELSE 0
            END
        ) AS null_pickup_datetime,
        SUM(
            CASE
                WHEN service_type = 'yellow' AND tpep_dropoff_datetime IS NULL THEN 1
                WHEN service_type = 'green'  AND lpep_dropoff_datetime IS NULL THEN 1
                ELSE 0
            END
        ) AS null_dropoff_datetime
    FROM raw_trips
    GROUP BY service_type
    ORDER BY service_type
""").df())

  service_type  total_rows  null_pickup_datetime  null_dropoff_datetime
0        green      591375                   0.0                    0.0
1       yellow    48722602                   0.0                    0.0


In [15]:
print(con.execute("""
    SELECT
        service_type,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS null_pu_location,
        SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS null_do_location,
        SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END) AS null_fare,
        SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS null_distance,
        SUM(CASE WHEN total_amount IS NULL THEN 1 ELSE 0 END) AS null_total
    FROM raw_trips
    GROUP BY service_type
    ORDER BY service_type
""").df())

  service_type  total_rows  null_pu_location  null_do_location  null_fare  \
0        green      591375               0.0               0.0        0.0   
1       yellow    48722602               0.0               0.0        0.0   

   null_distance  null_total  
0            0.0         0.0  
1            0.0         0.0  


In [16]:
print(con.execute("""
    SELECT
        service_type,
        DATE_TRUNC(
            'month',
            COALESCE(tpep_pickup_datetime, lpep_pickup_datetime)
        ) AS trip_month,
        COUNT(*) AS trip_count
    FROM raw_trips
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df())

   service_type trip_month  trip_count
0         green 2008-12-01           1
1         green 2024-12-01           6
2         green 2025-01-01       48288
3         green 2025-02-01       46645
4         green 2025-03-01       51550
5         green 2025-04-01       52134
6         green 2025-05-01       55405
7         green 2025-06-01       49385
8         green 2025-07-01       48202
9         green 2025-08-01       46305
10        green 2025-09-01       48885
11        green 2025-10-01       49417
12        green 2025-11-01       46915
13        green 2025-12-01       48223
14        green 2026-01-01          14
15       yellow 2007-12-01           1
16       yellow 2008-12-01           1
17       yellow 2009-01-01           6
18       yellow 2024-12-01          21
19       yellow 2025-01-01     3475234
20       yellow 2025-02-01     3577542
21       yellow 2025-03-01     4145229
22       yellow 2025-04-01     3970568
23       yellow 2025-05-01     4591844
24       yellow 2025-06-0

In [17]:
print(con.execute("""
    SELECT
        service_type,
        SUM(CASE WHEN trip_distance <= 0 THEN 1 ELSE 0 END) AS non_positive_distance,
        SUM(CASE WHEN fare_amount <= 0 THEN 1 ELSE 0 END) AS non_positive_fare,
        SUM(CASE WHEN total_amount <= 0 THEN 1 ELSE 0 END) AS non_positive_total,
        SUM(CASE WHEN passenger_count < 0 THEN 1 ELSE 0 END) AS negative_passenger_count
    FROM raw_trips
    GROUP BY service_type
    ORDER BY service_type
""").df())

  service_type  non_positive_distance  non_positive_fare  non_positive_total  \
0        green                24438.0             4469.0              2706.0   
1       yellow              1402958.0          2870234.0            980522.0   

   negative_passenger_count  
0                       0.0  
1                       0.0  


In [18]:
print(con.execute("""
    SELECT
        rt.service_type,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN z.LocationID IS NULL THEN 1 ELSE 0 END) AS unmatched_pickup_zones
    FROM raw_trips rt
    LEFT JOIN raw_zones z
        ON rt.PULocationID = z.LocationID
    GROUP BY rt.service_type
    ORDER BY rt.service_type
""").df())

  service_type  total_rows  unmatched_pickup_zones
0        green      591375                     0.0
1       yellow    48722602                     0.0


: 